 Phase 8–10 — Conserved Circuit Search
**Kaggle | T4 ×2 | Python 3.11**  
Loads the Phase 6-7 checkpoint, searches four candidate triplet families for the
largest directed induced subgraph conserved across at least 3 datasets, and
produces a valid submission CSV.


In [1]:
!pip install -q pandas==3.0.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 78.6 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have 

In [2]:
 ── STEP 0 — ENVIRONMENT ─────────────────────────────────────────────────
import subprocess, sys

def pip_install(pkg):
    try:
        __import__(pkg.split('[')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ['networkx', 'plotly', 'pandas', 'numpy', 'tqdm']:
    pip_install(pkg)

import gc, time, pickle, warnings, itertools, random
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
from tqdm import tqdm
warnings.filterwarnings('ignore')

 Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'networkx  {nx.__version__}')
print(f'pandas    {pd.__version__}')
print(f'numpy     {np.__version__}')

 ── Kaggle paths ─────────────────────────────────────────────────────────
 Primary path is the versioned Phase-5 dataset; fall back to
 the generic /kaggle/input/flywire mount if that path does not exist.
INPUT_DIR = Path('/kaggle/input/datasets/jit007/flywire-phase5')
if not INPUT_DIR.exists():
    INPUT_DIR = Path('/kaggle/input/flywire')
print('INPUT_DIR:', INPUT_DIR)

WORK_DIR       = Path('/kaggle/working')
EXPORT_DIR     = WORK_DIR / 'results' / 'exports'
CHECKPOINT_DIR = WORK_DIR / 'results' / 'checkpoints'
FIGURE_DIR     = WORK_DIR / 'results' / 'figures'
for d in [EXPORT_DIR, CHECKPOINT_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATASET_FILES = {
    'BANC': 'banc_626_edge_list.csv',
    'FAFB': 'fafb_783_edge_list.csv',
    'MANC': 'manc_1.2.1_edge_list.csv',
    'MAOL': 'maol_1.1_edge_list.csv',
    'MCNS': 'mcns_0.9_edge_list.csv',
}
PROCESS_ORDER = ['MANC', 'MAOL', 'BANC', 'FAFB', 'MCNS']

 Broader seed exploration with a tighter candidate cap per seed.
TOP_SEEDS             = 1000        seeds tested per family
MAX_CANDIDATES_TO_TEST = 200000    max candidates expanded per seed

print('Dirs ready. Input:', INPUT_DIR)


networkx  3.6.1
pandas    3.0.3
numpy     2.4.6
INPUT_DIR: /kaggle/input/datasets/jit007/flywire-phase5
Dirs ready. Input: /kaggle/input/datasets/jit007/flywire-phase5


In [3]:
 ── STEP 1 — LOAD PHASE 6-7 CHECKPOINT ─────────────────────────────────
print('=' * 60)
print('STEP 1 — LOAD PHASE 6-7 CHECKPOINT')
print('=' * 60)

ckpt_path = INPUT_DIR / 'phase6_7_checkpoint.pkl'
if not ckpt_path.exists():
    raise FileNotFoundError(f'Cannot find checkpoint at {ckpt_path}')

with open(ckpt_path, 'rb') as f:
    ckpt67 = pickle.load(f)

print('Keys found:', list(ckpt67.keys()))

REQUIRED = ['triplet_candidates', 'triplet_ranking',
            'candidate_matches', 'normalized_fingerprint_dfs']
for key in REQUIRED:
    if key not in ckpt67:
        raise KeyError(f'Missing required key in checkpoint: {key}')

triplet_candidates         = ckpt67['triplet_candidates']
triplet_ranking            = ckpt67['triplet_ranking']
candidate_matches          = ckpt67['candidate_matches']
normalized_fingerprint_dfs = ckpt67['normalized_fingerprint_dfs']

print(f'\nTriplet families found: {len(triplet_candidates)}')
for trip, df in triplet_candidates.items():
    print(f'  {str(trip):<35} seeds={len(df):>10,}')
print('\ntriplet_ranking shape:', triplet_ranking.shape)
print('candidate_matches keys:', list(candidate_matches.keys()))
print('Phase 6-7 checkpoint loaded and validated')



STEP 1 — LOAD PHASE 6-7 CHECKPOINT
Keys found: ['hub_nodes', 'hub_thresholds', 'fingerprint_dfs', 'wl_label_dfs', 'normalized_fingerprint_dfs', 'candidate_matches', 'triplet_candidates', 'triplet_ranking', 'wl_rarity_dfs', 'topology_dfs', 'phase6_7_config']

Triplet families found: 10
  ('MANC', 'MAOL', 'BANC')            seeds=   236,410
  ('MANC', 'MAOL', 'FAFB')            seeds=   236,410
  ('MANC', 'MAOL', 'MCNS')            seeds=   236,410
  ('MANC', 'BANC', 'FAFB')            seeds=   236,410
  ('MANC', 'BANC', 'MCNS')            seeds=   236,410
  ('MANC', 'FAFB', 'MCNS')            seeds=   236,410
  ('MAOL', 'BANC', 'FAFB')            seeds=   516,690
  ('MAOL', 'BANC', 'MCNS')            seeds=   516,690
  ('MAOL', 'FAFB', 'MCNS')            seeds=   516,690
  ('BANC', 'FAFB', 'MCNS')            seeds= 1,128,850

triplet_ranking shape: (10, 5)
candidate_matches keys: ['MANC_MAOL', 'MANC_BANC', 'MANC_FAFB', 'MANC_MCNS', 'MAOL_BANC', 'MAOL_FAFB', 'MAOL_MCNS', 'BANC_FAFB', 'BA

In [4]:
trip = ('MANC', 'FAFB', 'MCNS')

df = triplet_candidates[trip]

v = df.iloc[0]['node_2']

print(v)
print(repr(v))
print(type(v))

720575940635468991
'720575940635468991'
<class 'str'>


In [5]:
 ── STEP 2 — LOAD ORIGINAL CONNECTOMES ─────────────────────────────────
print('=' * 60)
print('STEP 2 — LOAD ORIGINAL CONNECTOMES')
print('=' * 60)

graphs = {}
for name in PROCESS_ORDER:
    t0 = time.time()
    fpath = INPUT_DIR / DATASET_FILES[name]
    if not fpath.exists():
        raise FileNotFoundError(f'Missing edge list: {fpath}')
    df_e = pd.read_csv(fpath, usecols=[0, 1], header=0,
                       dtype={'pre_root_id': np.int64, 'post_root_id': np.int64})
    df_e.columns = ['src', 'dst']
    G = nx.from_pandas_edgelist(df_e, 'src', 'dst', create_using=nx.DiGraph())
    graphs[name] = G
    print(f'  {name:<6}  nodes={G.number_of_nodes():>8,}  '
          f'edges={G.number_of_edges():>10,}  [{time.time()-t0:.1f}s]')
    del df_e; gc.collect()

print('All graphs loaded')

 ── Node type audit — detect type mismatches before circuit growth ─────
print('Graph node types:')
for name, G in graphs.items():
    sample_node = next(iter(G.nodes()))
    print(f'  {name:<6}  node type: {type(sample_node).__name__}')

_sample_family = next(iter(triplet_candidates))
_sample_df     = triplet_candidates[_sample_family]
if len(_sample_df) > 0:
    _row = _sample_df.iloc[0]
    print(f'\nTriplet node types (family {_sample_family}):')
    print(f'  node_1 type: {type(_row["node_1"]).__name__}')
    print(f'  node_2 type: {type(_row["node_2"]).__name__}')
    print(f'  node_3 type: {type(_row["node_3"]).__name__}')

 ── Per-family, per-column type conversion ───────────────────────────────
 Each dataset in a triplet may have a different node ID type.
 Convert each column to match the type used by its corresponding graph.
_any_converted = False
for trip_key, df in triplet_candidates.items():
    d1_name, d2_name, d3_name = trip_key
    type1 = type(next(iter(graphs[d1_name].nodes())))
    type2 = type(next(iter(graphs[d2_name].nodes())))
    type3 = type(next(iter(graphs[d3_name].nodes())))

    _row = df.iloc[0] if len(df) > 0 else None
    if _row is None:
        continue

    needs = (
        not isinstance(_row['node_1'], type1) or
        not isinstance(_row['node_2'], type2) or
        not isinstance(_row['node_3'], type3)
    )
    if needs:
        df['node_1'] = df['node_1'].apply(type1)
        df['node_2'] = df['node_2'].apply(type2)
        df['node_3'] = df['node_3'].apply(type3)
        triplet_candidates[trip_key] = df
        _any_converted = True

if _any_converted:
    print('\nType conversion applied')
else:
    print('\nNo conversion needed')
print()

 ── Build pair lookups AFTER type conversion ─────────────────────────────
 candidate_matches must share the same node ID type as the graphs.
 Convert candidate_matches columns using the same per-dataset type map.
_ds_type = {name: type(next(iter(G.nodes()))) for name, G in graphs.items()}

_cm_converted = False
for key, df in candidate_matches.items():
    parts = key.split('_', 1)
    if len(parts) == 2 and parts[0] in _ds_type and parts[1] in _ds_type:
        t1 = _ds_type[parts[0]]
        t2 = _ds_type[parts[1]]
        _r = df.iloc[0] if len(df) > 0 else None
        if _r is not None and (
            not isinstance(_r['node_1'], t1) or
            not isinstance(_r['node_2'], t2)
        ):
            candidate_matches[key]['node_1'] = df['node_1'].apply(t1)
            candidate_matches[key]['node_2'] = df['node_2'].apply(t2)
            _cm_converted = True

pair_lookup = {}
for key, df in candidate_matches.items():
    pair_lookup[key] = set(zip(df['node_1'], df['node_2']))
    print(f'  pair_lookup[{key!r}] {len(pair_lookup[key]):,} pairs')
print()


STEP 2 — LOAD ORIGINAL CONNECTOMES
  MANC    nodes=  23,641  edges= 5,305,638  [14.5s]
  MAOL    nodes=  51,669  edges= 6,484,936  [20.1s]
  BANC    nodes= 112,885  edges= 2,676,592  [13.7s]
  FAFB    nodes= 138,584  edges= 3,732,460  [15.2s]
  MCNS    nodes= 165,820  edges= 6,239,112  [21.2s]
All graphs loaded
Graph node types:
  MANC    node type: int
  MAOL    node type: int
  BANC    node type: int
  FAFB    node type: int
  MCNS    node type: int

Triplet node types (family ('MANC', 'MAOL', 'BANC')):
  node_1 type: str
  node_2 type: str
  node_3 type: str

Type conversion applied

  pair_lookup['MANC_MAOL'] 1,182,050 pairs
  pair_lookup['MANC_BANC'] 1,182,050 pairs
  pair_lookup['MANC_FAFB'] 1,182,050 pairs
  pair_lookup['MANC_MCNS'] 1,182,050 pairs
  pair_lookup['MAOL_BANC'] 2,583,450 pairs
  pair_lookup['MAOL_FAFB'] 2,583,450 pairs
  pair_lookup['MAOL_MCNS'] 2,583,450 pairs
  pair_lookup['BANC_FAFB'] 5,644,250 pairs
  pair_lookup['BANC_MCNS'] 5,644,250 pairs
  pair_lookup['FAFB

In [6]:
 Build adjacency sets once — reused everywhere, never rebuilt inside loops
print('Building adjacency sets...')
adj_sets = {}
for name, G in graphs.items():
    t0 = time.time()
    adj_sets[name] = set(G.edges())
    print(f'  {name:<6}  adj_set size={len(adj_sets[name]):>10,}  [{time.time()-t0:.1f}s]')

print('\n✓ Adjacency sets ready (reused for all lookups)')


Building adjacency sets...
  MANC    adj_set size= 5,305,638  [5.4s]
  MAOL    adj_set size= 6,484,936  [2.8s]
  BANC    adj_set size= 2,676,592  [1.3s]
  FAFB    adj_set size= 3,732,460  [1.7s]
  MCNS    adj_set size= 6,239,112  [2.9s]

✓ Adjacency sets ready (reused for all lookups)


In [7]:
 ── STEP 3 — SELECT CANDIDATE FAMILIES ─────────────────────────────────
print('=' * 60)
print('STEP 3 — SELECT CANDIDATE FAMILIES')
print('=' * 60)




 Fast test using the top 3 ranked families
TARGET_FAMILIES = [
    ('MANC', 'MAOL', 'MCNS'),
    ('BANC', 'FAFB', 'MCNS'),
    ('MAOL', 'FAFB', 'MCNS'),
    ('MAOL', 'BANC', 'MCNS'),
    ('MANC', 'FAFB', 'MCNS'),
]
family_seeds = {}

for family in TARGET_FAMILIES:

    df_seeds    = None
    found_perm  = None   the permutation actually stored in the checkpoint

     Phase 6-7 may have stored triplets under a different ordering.
    for perm in itertools.permutations(family):
        if perm in triplet_candidates:
            df_seeds   = triplet_candidates[perm].copy()
            found_perm = perm
            break

     Diagnostic: show whether stored ordering matches requested family.
    if found_perm is not None:
        reorder_needed = (found_perm != family)
        print(f'  Requested family : {family}')
        print(f'  Stored family    : {found_perm}')
        print(f'  Column reorder   : {"YES" if reorder_needed else "NO"}')
        print()

    if df_seeds is None or df_seeds.empty:
        print(f'  WARNING: no seeds found for {family} — skipping')
        continue

     Reorder node columns so node_1/2/3 always map to family[0/1/2].
     Explicit reconstruction avoids cyclic rename collisions.
    if found_perm != family:
         Map each dataset name to the column that currently holds it.
        ds_to_col = {found_perm[i]: f'node_{i+1}' for i in range(3)}
        node_cols  = ['node_1', 'node_2', 'node_3']
        other_cols = [c for c in df_seeds.columns if c not in node_cols]
        df_reordered = pd.DataFrame({
            'node_1': df_seeds[ds_to_col[family[0]]].values,
            'node_2': df_seeds[ds_to_col[family[1]]].values,
            'node_3': df_seeds[ds_to_col[family[2]]].values,
        })
        for col in other_cols:
            df_reordered[col] = df_seeds[col].values
        df_seeds = df_reordered

     Lower triplet_score = better seed
    df_seeds = (
        df_seeds
        .sort_values('triplet_score', ascending=True)
        .reset_index(drop=True)
    )

    print("\nTop-ranked seed diagnostics:")

    diag_cols = [
        c for c in [
            "triplet_score",
            "ac_supported",
            "similarity_score"
        ]
        if c in df_seeds.columns
    ]

    print(
        df_seeds[diag_cols]
        .head(10)
    )

     ------------------------------------------------------------------
     Force node IDs to match graph node types
     ------------------------------------------------------------------
    d1, d2, d3 = family

    type1 = type(next(iter(graphs[d1].nodes())))
    type2 = type(next(iter(graphs[d2].nodes())))
    type3 = type(next(iter(graphs[d3].nodes())))

    df_seeds['node_1'] = df_seeds['node_1'].astype(type1)
    df_seeds['node_2'] = df_seeds['node_2'].astype(type2)
    df_seeds['node_3'] = df_seeds['node_3'].astype(type3)

     Verify conversion worked
    print('\nTop seed examples:')
    print(
        df_seeds[
            ['node_1', 'node_2', 'node_3', 'similarity_score']
        ].head(5)
    )

    print(
        'Node types:',
        type(df_seeds.iloc[0]['node_1']).__name__,
        type(df_seeds.iloc[0]['node_2']).__name__,
        type(df_seeds.iloc[0]['node_3']).__name__
    )

    family_seeds[family] = df_seeds

    print(
        f'  Family {str(family):<35} '
        f'seeds={len(df_seeds):>10,}'
    )
    print()

 Force integer node IDs before search

for family_key, df in family_seeds.items():
    df['node_1'] = pd.to_numeric(df['node_1']).astype(np.int64)
    df['node_2'] = pd.to_numeric(df['node_2']).astype(np.int64)
    df['node_3'] = pd.to_numeric(df['node_3']).astype(np.int64)

    family_seeds[family_key] = df

print('\nFinal Data Types:')
if family_seeds:
    first_fam = next(iter(family_seeds.values()))
    print(first_fam[['node_1', 'node_2', 'node_3']].dtypes)

sample_family = next(iter(family_seeds))
sample_df = family_seeds[sample_family]

print("\nSample node values:")
print(sample_df[['node_1','node_2','node_3']].head())

assert sample_df['node_1'].dtype == np.int64
assert sample_df['node_2'].dtype == np.int64
assert sample_df['node_3'].dtype == np.int64

if not family_seeds:
    raise RuntimeError(
        'No families could be loaded — check checkpoint keys'
    )

print(f'{len(family_seeds)} families ready for search')

STEP 3 — SELECT CANDIDATE FAMILIES
  Requested family : ('MANC', 'MAOL', 'MCNS')
  Stored family    : ('MANC', 'MAOL', 'MCNS')
  Column reorder   : NO


Top-ranked seed diagnostics:
   triplet_score  ac_supported  similarity_score
0      -0.244251          True         -0.244251
1      -0.242972          True         -0.242972
2      -0.242923          True         -0.242923
3      -0.242710          True         -0.242710
4      -0.242628          True         -0.242628
5      -0.242495          True         -0.242495
6      -0.242172          True         -0.242172
7      -0.241941          True         -0.241941
8      -0.241814          True         -0.241814
9      -0.241565          True         -0.241565

Top seed examples:
   node_1  node_2  node_3  similarity_score
0  157964   58690   15114         -0.244251
1  157964   58690   16238         -0.242972
2  157861  573615   24016         -0.242923
3  164377   78624   14372         -0.242710
4  164377   63296   31124         -0.24

In [8]:
 ── STEP 4 — CORRESPONDENCE VALIDATOR ─────────────────────────────
 Core validation logic for circuit growth.
 Uses adjacency-set lookups (O(1)).
 A candidate is accepted only if:
   1. Fixed correspondence holds across all datasets.
   2. The candidate participates in at least one conserved edge.

def validate_candidate(current_mapping, c1, c2, c3, ds_names, adj_sets):
    """
    Validate whether candidate triplet (c1, c2, c3) can be added to the
    current conserved mapping.

    Conditions:
      - Edge presence/absence must match across all datasets.
      - Candidate must participate in at least one conserved edge.
    """

    d1, d2, d3 = ds_names

    a1 = adj_sets[d1]
    a2 = adj_sets[d2]
    a3 = adj_sets[d3]

    has_conserved_edge = False

    for (t1, t2, t3) in current_mapping:

         ----------------------------------------------------------
         Existing -> Candidate
         ----------------------------------------------------------
        e1_fwd = (t1, c1) in a1
        e2_fwd = (t2, c2) in a2
        e3_fwd = (t3, c3) in a3

        if not (e1_fwd == e2_fwd == e3_fwd):
            return False

        if e1_fwd:
            has_conserved_edge = True

         ----------------------------------------------------------
         Candidate -> Existing
         ----------------------------------------------------------
        e1_bwd = (c1, t1) in a1
        e2_bwd = (c2, t2) in a2
        e3_bwd = (c3, t3) in a3

        if not (e1_bwd == e2_bwd == e3_bwd):
            return False

        if e1_bwd:
            has_conserved_edge = True

     Reject isolated candidates
    return has_conserved_edge


 Accept the pre-computed triplet pool from Phase 6-7 directly.
def grow_circuit_from_seed(
    seed_c1,
    seed_c2,
    seed_c3,
    family,
    family_pool,
    adj_sets,
    MAX_CANDIDATES
):
    """
    Greedy circuit growth from a seed triplet.

    Returns:
        List of conserved triplets.
    """

    d1, d2, d3 = family

    mapping = [(int(seed_c1), int(seed_c2), int(seed_c3))]

    in_mapping = {
        d1: {int(seed_c1)},
        d2: {int(seed_c2)},
        d3: {int(seed_c3)},
    }

    pool = family_pool.head(MAX_CANDIDATES)

    for row in pool.itertuples(index=False):

        c1 = int(row.node_1)
        c2 = int(row.node_2)
        c3 = int(row.node_3)

         Prevent duplicate neuron usage
        if (
            c1 in in_mapping[d1]
            or c2 in in_mapping[d2]
            or c3 in in_mapping[d3]
        ):
            continue

        if validate_candidate(
            mapping,
            c1,
            c2,
            c3,
            family,
            adj_sets
        ):
            mapping.append((c1, c2, c3))

            in_mapping[d1].add(c1)
            in_mapping[d2].add(c2)
            in_mapping[d3].add(c3)

    return mapping


print('✓ Validation functions defined')

✓ Validation functions defined


In [9]:
 ── STEP 4 — MULTI-START CIRCUIT SEARCH ─────────────────────────────────
print('=' * 60)
print('STEP 4 — MULTI-START CIRCUIT SEARCH')
print('=' * 60)

best_mapping    = []
best_N          = 0
best_family     = None
best_seed_rank  = None
family_results  = []    summary across all families for comparison plot

print("\nStored triplet keys:")
for k in sorted(triplet_candidates.keys()):
    print(k)
    

for family in TARGET_FAMILIES:
    if family not in family_seeds:
        continue

    df_seeds = family_seeds[family]
    n_seeds  = min(TOP_SEEDS, len(df_seeds))
    d1, d2, d3 = family

     Pass the full sorted pool so grow_circuit_from_seed uses Phase 6-7 triplets.
    family_pool = df_seeds   already sorted ascending by triplet_score

    print(f'\nFamily {d1}-{d2}-{d3}  —  testing top {n_seeds} seeds')
    family_best_n   = 0
    family_best_map = []
    family_sizes    = []    for seed-size plot

    for seed_rank in tqdm(range(n_seeds), desc=f'{d1}-{d2}-{d3}', leave=False):
        c1 = int(df_seeds['node_1'].values[seed_rank])
        c2 = int(df_seeds['node_2'].values[seed_rank])
        c3 = int(df_seeds['node_3'].values[seed_rank])
        mapping = grow_circuit_from_seed(
            c1, c2, c3,
            family, family_pool, adj_sets, MAX_CANDIDATES_TO_TEST
        )
        n = len(mapping)
        family_sizes.append({'seed_rank': seed_rank, 'circuit_size': n})

        if n > family_best_n:
            family_best_n   = n
            family_best_map = mapping

     Update global best
    if family_best_n > best_N:
        best_N         = family_best_n
        best_mapping   = family_best_map
        best_family    = family
        best_seed_rank = next(
            (r['seed_rank'] for r in family_sizes
             if r['circuit_size'] == family_best_n), 0)

     Record summary statistics for this family.
    sizes_only = [x['circuit_size'] for x in family_sizes]
    family_results.append({
        'family'     : f'{d1}-{d2}-{d3}',
        'best_size'  : family_best_n,
        'mean_size'  : float(np.mean(sizes_only)),
        'median_size': float(np.median(sizes_only)),
        'max_size'   : family_best_n,
        'seed_sizes' : family_sizes,
    })
    print(f'  Best for {d1}-{d2}-{d3}: {family_best_n} neurons')

     Checkpoint after each family search
    with open(CHECKPOINT_DIR / 'phase8_10_checkpoint_partial.pkl', 'wb') as f:
        pickle.dump({
            'best_mapping' : best_mapping,
            'best_N'       : best_N,
            'best_family'  : best_family,
            'best_seed_rank': best_seed_rank,
            'family_results': family_results,
        }, f)

print(f'\n✓ Search complete — global best: {best_N} neurons  family={best_family}')


STEP 4 — MULTI-START CIRCUIT SEARCH

Stored triplet keys:
('BANC', 'FAFB', 'MCNS')
('MANC', 'BANC', 'FAFB')
('MANC', 'BANC', 'MCNS')
('MANC', 'FAFB', 'MCNS')
('MANC', 'MAOL', 'BANC')
('MANC', 'MAOL', 'FAFB')
('MANC', 'MAOL', 'MCNS')
('MAOL', 'BANC', 'FAFB')
('MAOL', 'BANC', 'MCNS')
('MAOL', 'FAFB', 'MCNS')

Family MANC-MAOL-MCNS  —  testing top 1000 seeds


  Best for MANC-MAOL-MCNS: 9 neurons

Family BANC-FAFB-MCNS  —  testing top 1000 seeds


  Best for BANC-FAFB-MCNS: 44 neurons

Family MAOL-FAFB-MCNS  —  testing top 1000 seeds


  Best for MAOL-FAFB-MCNS: 126 neurons

Family MAOL-BANC-MCNS  —  testing top 1000 seeds


  Best for MAOL-BANC-MCNS: 5 neurons

Family MANC-FAFB-MCNS  —  testing top 1000 seeds


  Best for MANC-FAFB-MCNS: 6 neurons

✓ Search complete — global best: 126 neurons  family=('MAOL', 'FAFB', 'MCNS')


In [10]:
 ── STEP 4B — SECOND PASS EXPANSION ─────────────────────────────────────
 Additional refinement stage: scan ALL remaining triplets in the winning
 family pool (beyond the TOP_SEEDS window) and greedily add any that pass

 this only extends best_mapping.
print('=' * 60)
print('STEP 4B — SECOND PASS EXPANSION')
print('=' * 60)

 Use the winning family only
family_pool = family_seeds[best_family]
d1, d2, d3  = best_family

 Build a set of all triplets already inside best_mapping for fast skip
in_best = set(best_mapping)          set of (n1, n2, n3) tuples
in_d1   = set(t[0] for t in best_mapping)
in_d2   = set(t[1] for t in best_mapping)
in_d3   = set(t[2] for t in best_mapping)

added_count = 0

 Iterate through ALL candidates in ascending triplet_score order.
 family_pool is already sorted from Step 3.
for row in tqdm(
    family_pool.itertuples(index=False),
    total=len(family_pool),
    desc='2nd-pass expansion',
    leave=False
):
    candidate = (
        int(row.node_1),
        int(row.node_2),
        int(row.node_3)
    )

     Skip if this exact triplet is already accepted
    if candidate in in_best:
        continue

     Skip if any individual node is already used (no duplicates)
    if (candidate[0] in in_d1 or
        candidate[1] in in_d2 or
        candidate[2] in in_d3):
        continue

     Run fixed-correspondence validation
    if validate_candidate(best_mapping,
                          candidate[0], candidate[1], candidate[2],
                          best_family, adj_sets):
        best_mapping.append(candidate)
        in_best.add(candidate)
        in_d1.add(candidate[0])
        in_d2.add(candidate[1])
        in_d3.add(candidate[2])
        added_count += 1

best_N = len(best_mapping)
print(f'Second-pass expansion added {added_count} neurons')
print(f'Updated best_N = {best_N}')

best_mapping = [
    (int(a), int(b), int(c))
    for a, b, c in best_mapping
]

print(type(best_mapping[0][0]))
print(type(best_mapping[0][1]))
print(type(best_mapping[0][2]))


STEP 4B — SECOND PASS EXPANSION


Second-pass expansion added 35 neurons
Updated best_N = 161
<class 'int'>
<class 'int'>
<class 'int'>


In [11]:
 ── STEP 4C — REVERIFY REFINED CIRCUIT ──────────────────────────────────
 The second-pass expansion modifies best_mapping.
 Re-run fixed-correspondence verification before metrics are computed.
import time

print('=' * 60)
print('STEP 4C — REVERIFY REFINED CIRCUIT')
print('=' * 60)

from tqdm import tqdm

def verify_fixed(mapping, ds_names, adj_sets):
    """
    For every unordered pair of triplets (i, j) in the mapping, verify
    that the directed edge pattern is identical across all three datasets
    in BOTH directions (i→j and j→i).

    mapping:  list of (n1, n2, n3) tuples
    Returns:  True if all pairs are consistent, raises RuntimeError otherwise.
    """
    dn1, dn2, dn3 = ds_names
    a1 = adj_sets[dn1]
    a2 = adj_sets[dn2]
    a3 = adj_sets[dn3]
    n = len(mapping)
    for i in tqdm(range(n), desc="Verifying Pairs"):
        ai, bi, ci = mapping[i]
        for j in range(i + 1, n):
            aj, bj, cj = mapping[j]
             ── direction i → j ──────────────────────────────────
            e1_fwd = (ai, aj) in a1
            e2_fwd = (bi, bj) in a2
            e3_fwd = (ci, cj) in a3
            if not (e1_fwd == e2_fwd == e3_fwd):
                raise RuntimeError(
                    f'Fixed-correspondence FAILED at pair (i={i}, j={j}) '
                    f'direction i→j: G1={e1_fwd}  G2={e2_fwd}  G3={e3_fwd}'
                )
             ── direction j → i ──────────────────────────────────
            e1_rev = (aj, ai) in a1
            e2_rev = (bj, bi) in a2
            e3_rev = (cj, ci) in a3
            if not (e1_rev == e2_rev == e3_rev):
                raise RuntimeError(
                    f'Fixed-correspondence FAILED at pair (i={i}, j={j}) '
                    f'direction j→i: G1={e1_rev}  G2={e2_rev}  G3={e3_rev}'
                )
    return True


 ── Bijection sanity check ────────────────────────────────────────
total_triplets = len(best_mapping)
nodes1 = len(set(t[0] for t in best_mapping))
nodes2 = len(set(t[1] for t in best_mapping))
nodes3 = len(set(t[2] for t in best_mapping))

print(f'  Mapped triplets : {total_triplets}')
print(f'  Unique nodes — dataset 1: {nodes1}, dataset 2: {nodes2}, dataset 3: {nodes3}')

if not (total_triplets == nodes1 == nodes2 == nodes3):
    raise ValueError(
        f'Mapping is NOT a strict bijection: '
        f'triplets={total_triplets}, ds1={nodes1}, ds2={nodes2}, ds3={nodes3}'
    )
print('  ✓ Bijection check passed — mapping is one-to-one across all datasets')

 ── Run verification with timing ──────────────────────────────────
_t0 = time.time()
verify_fixed(
    best_mapping,
    best_family,
    adj_sets
)
_elapsed = time.time() - _t0
print(f'✓ Refined circuit passed fixed-correspondence verification')
print(f'  Verification took {_elapsed:.1f} seconds')

 Edge sanity check

d1, d2, d3 = best_family

G1_check = graphs[d1].subgraph([t[0] for t in best_mapping])
G2_check = graphs[d2].subgraph([t[1] for t in best_mapping])
G3_check = graphs[d3].subgraph([t[2] for t in best_mapping])

e1 = G1_check.number_of_edges()
e2 = G2_check.number_of_edges()
e3 = G3_check.number_of_edges()

print()
print('Edge sanity check')
print('G1 edges:', e1)
print('G2 edges:', e2)
print('G3 edges:', e3)

if max(e1, e2, e3) == 0:
    raise RuntimeError(
        'Circuit contains zero edges. Search found isolated nodes only.'
    )

if not (e1 == e2 == e3):
    raise RuntimeError(
        f'VALIDATOR FAILURE: Edge count mismatch! '
        f'{d1}={e1}, {d2}={e2}, {d3}={e3}. '
        f'The growth logic admitted a structurally inconsistent node.'
    )

print("\nMapping type audit:")
print(type(best_mapping[0][0]))
print(type(best_mapping[0][1]))
print(type(best_mapping[0][2]))

assert isinstance(best_mapping[0][0], int)
assert isinstance(best_mapping[0][1], int)
assert isinstance(best_mapping[0][2], int)

STEP 4C — REVERIFY REFINED CIRCUIT
  Mapped triplets : 161
  Unique nodes — dataset 1: 161, dataset 2: 161, dataset 3: 161
  ✓ Bijection check passed — mapping is one-to-one across all datasets


Verifying Pairs: 100%|██████████| 161/161 [00:00<00:00, 6021.68it/s]

✓ Refined circuit passed fixed-correspondence verification
  Verification took 0.0 seconds

Edge sanity check
G1 edges: 301
G2 edges: 301
G3 edges: 301

Mapping type audit:
<class 'int'>
<class 'int'>
<class 'int'>


In [12]:
 ── STEP 5 — COMPONENT ANALYSIS ─────────────────────────────────────────
print('=' * 60)
print('STEP 5 — COMPONENT ANALYSIS')
print('=' * 60)

d1, d2, d3 = best_family

nodes_d1 = [t[0] for t in best_mapping]
nodes_d2 = [t[1] for t in best_mapping]
nodes_d3 = [t[2] for t in best_mapping]

 Induced subgraph on the primary dataset (d1) for metrics
G_sub = graphs[d1].subgraph(nodes_d1).copy()

n_nodes   = G_sub.number_of_nodes()
n_edges   = G_sub.number_of_edges()
max_edges = n_nodes * (n_nodes - 1) if n_nodes > 1 else 1
density   = n_edges / max_edges if max_edges > 0 else 0
avg_deg   = n_edges / n_nodes if n_nodes > 0 else 0

in_degrees  = [d for _, d in G_sub.in_degree()]
out_degrees = [d for _, d in G_sub.out_degree()]

metrics = {
    'family'      : f'{d1}-{d2}-{d3}',
    'seed_rank'   : best_seed_rank,
    'nodes'       : n_nodes,
    'edges'       : n_edges,
    'density'     : round(density, 6),
    'avg_degree'  : round(avg_deg, 4),
    'max_in_degree'  : max(in_degrees)  if in_degrees  else 0,
    'max_out_degree' : max(out_degrees) if out_degrees else 0,
    'weak_components': nx.number_weakly_connected_components(G_sub),
}

metrics_df = pd.DataFrame([metrics])
metrics_df.to_csv(EXPORT_DIR / 'circuit_metrics.csv', index=False)

print('Circuit metrics:')
for k, v in metrics.items():
    print(f'  {k:<20}: {v}')
print('\n✓ Metrics exported')


STEP 5 — COMPONENT ANALYSIS
Circuit metrics:
  family              : MAOL-FAFB-MCNS
  seed_rank           : 9
  nodes               : 161
  edges               : 301
  density             : 0.011685
  avg_degree          : 1.8696
  max_in_degree       : 153
  max_out_degree      : 148
  weak_components     : 1

✓ Metrics exported


In [13]:
print(best_family)

print(best_mapping[:5])

print(type(best_mapping[0][0]))
print(type(best_mapping[0][1]))
print(type(best_mapping[0][2]))

print(type(next(iter(graphs['MAOL'].nodes()))))
print(type(next(iter(graphs['BANC'].nodes()))))
print(type(next(iter(graphs['FAFB'].nodes()))))

('MAOL', 'FAFB', 'MCNS')
[(10009, 720575940628908548, 10157), (104110, 720575940620889377, 145440), (115921, 720575940644403656, 116841), (118225, 720575940629228905, 108243), (120351, 720575940618527515, 163055)]
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>
<class 'int'>


In [14]:
 ── STEP 6 — EXACT ISOMORPHISM VERIFICATION ─────────────────────────────
 This step extracts induced subgraphs, prints statistics, relabels nodes,
 then runs DiGraphMatcher for structural isomorphism validation.
print('=' * 60)
print('STEP 6 — EXACT ISOMORPHISM VERIFICATION')
print('=' * 60)

from networkx.algorithms import isomorphism

d1, d2, d3 = best_family

nodes_d1 = [t[0] for t in best_mapping]
nodes_d2 = [t[1] for t in best_mapping]
nodes_d3 = [t[2] for t in best_mapping]

G1 = graphs[d1].subgraph(nodes_d1).copy()
G2 = graphs[d2].subgraph(nodes_d2).copy()
G3 = graphs[d3].subgraph(nodes_d3).copy()

print(f'G1 ({d1}): {G1.number_of_nodes()} nodes, {G1.number_of_edges()} edges')
print(f'G2 ({d2}): {G2.number_of_nodes()} nodes, {G2.number_of_edges()} edges')
print(f'G3 ({d3}): {G3.number_of_nodes()} nodes, {G3.number_of_edges()} edges')

 Track validation outcomes; fixed_correspondence was already confirmed above.
validation_results = {
    'fixed_correspondence': True
}

 ── DiGraphMatcher (structural isomorphism) ──────────────────────
 Build relabeled copies using position index for isomorphism check
 (DiGraphMatcher works on node labels, so we relabel by index)
print('\nStage 2 — DiGraphMatcher...')
idx1 = {n: i for i, n in enumerate(nodes_d1)}
idx2 = {n: i for i, n in enumerate(nodes_d2)}
idx3 = {n: i for i, n in enumerate(nodes_d3)}

G1r = nx.relabel_nodes(G1, idx1)
G2r = nx.relabel_nodes(G2, idx2)
G3r = nx.relabel_nodes(G3, idx3)

matcher12 = isomorphism.DiGraphMatcher(G1r, G2r)
iso12 = matcher12.is_isomorphic()
validation_results['G1_vs_G2'] = iso12
print(f'  Isomorphic G1 vs G2: {iso12}')

matcher13 = isomorphism.DiGraphMatcher(G1r, G3r)
iso13 = matcher13.is_isomorphic()
validation_results['G1_vs_G3'] = iso13
print(f'  Isomorphic G1 vs G3: {iso13}')

if not iso12 or not iso13:
    raise RuntimeError(
        'VALIDATION FAILED: induced subgraphs are not isomorphic. '
        'The mapping may be inconsistent. Review grow_circuit_from_seed.'
    )

print('\n✓ EXACT ISOMORPHISM VERIFIED')

print('\nWeak connectivity check')

wc1 = nx.is_weakly_connected(G1)
wc2 = nx.is_weakly_connected(G2)
wc3 = nx.is_weakly_connected(G3)

print(f'  {d1}: {wc1}')
print(f'  {d2}: {wc2}')
print(f'  {d3}: {wc3}')

print(
    f'  Components: '
    f'{nx.number_weakly_connected_components(G1)}, '
    f'{nx.number_weakly_connected_components(G2)}, '
    f'{nx.number_weakly_connected_components(G3)}'
)

if not (wc1 and wc2 and wc3):
    raise RuntimeError(
        'FlyWire requirement failed: circuit is not weakly connected.'
    )

best_edges = G1.number_of_edges()

STEP 6 — EXACT ISOMORPHISM VERIFICATION
G1 (MAOL): 161 nodes, 301 edges
G2 (FAFB): 161 nodes, 301 edges
G3 (MCNS): 161 nodes, 301 edges

Stage 2 — DiGraphMatcher...
  Isomorphic G1 vs G2: True
  Isomorphic G1 vs G3: True

✓ EXACT ISOMORPHISM VERIFIED

Weak connectivity check
  MAOL: True
  FAFB: True
  MCNS: True
  Components: 1, 1, 1


In [15]:
 ── STEP 7 — SUBMISSION FILE ─────────────────────────────────────────────
print('=' * 60)
print('STEP 7 — CREATING SUBMISSION FILE')
print('=' * 60)

d1, d2, d3 = best_family

 Use actual dataset names as column headers — the FlyWire scorer
 maps column names directly to connectome datasets.
submission_df = pd.DataFrame(best_mapping, columns=[d1, d2, d3])

 Validation: no duplicate neuron IDs in any column
for col in [d1, d2, d3]:
    if submission_df[col].duplicated().any():
        raise ValueError(f'DUPLICATE neurons found in column {col}')

submission_path = WORK_DIR / 'submission.csv'
submission_df.to_csv(submission_path, index=False)

print(f'Submission rows   : {len(submission_df):,}')
print(f'Column names      : {list(submission_df.columns)}')
print(f'Saved to          : {submission_path}')
print(submission_df.head(5).to_string(index=False))
print('\n✓ submission.csv ready')


STEP 7 — CREATING SUBMISSION FILE
Submission rows   : 161
Column names      : ['MAOL', 'FAFB', 'MCNS']
Saved to          : /kaggle/working/submission.csv
  MAOL               FAFB   MCNS
 10009 720575940628908548  10157
104110 720575940620889377 145440
115921 720575940644403656 116841
118225 720575940629228905 108243
120351 720575940618527515 163055

✓ submission.csv ready


In [16]:
 ── STEP 8 — VISUALIZATIONS ─────────────────────────────────────────────
print('=' * 60)
print('STEP 8 — GENERATING FIGURES')
print('=' * 60)

d1, d2, d3 = best_family

 ── Figure 1: Conserved circuit network (spring layout, node=50 limit) ─
display_G = G1.subgraph(nodes_d1[:min(50, len(nodes_d1))]).copy()
pos = nx.spring_layout(display_G, seed=RANDOM_SEED)
edge_x, edge_y = [], []
for u, v in display_G.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]
node_x = [pos[n][0] for n in display_G.nodes()]
node_y = [pos[n][1] for n in display_G.nodes()]
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.8, color='666'), hoverinfo='none'))
fig1.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers',
    marker=dict(size=8, color='00d4ff', line=dict(width=1, color='white')),
    text=[str(n) for n in display_G.nodes()], hoverinfo='text'))
fig1.update_layout(title=f'Conserved Circuit — {d1}/{d2}/{d3} (first 50 nodes)',
    showlegend=False, template='plotly_dark',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
fig1.write_html(str(FIGURE_DIR / 'circuit_network.html'))
print('  ✓ circuit_network.html')

 ── Figure 2: Total degree histogram ────────────────────────────────────
total_deg = [d for _, d in G1.degree()]
fig2 = px.histogram(x=total_deg, nbins=30,
    title='Circuit — Total Degree Distribution', template='plotly_dark',
    labels={'x': 'Total Degree', 'y': 'Count'})
fig2.write_html(str(FIGURE_DIR / 'degree_histogram.html'))
print('  ✓ degree_histogram.html')

 ── Figure 3: In-degree histogram ───────────────────────────────────────
fig3 = px.histogram(x=in_degrees, nbins=30,
    title='Circuit — In-Degree Distribution', template='plotly_dark',
    labels={'x': 'In-Degree', 'y': 'Count'})
fig3.write_html(str(FIGURE_DIR / 'in_degree_histogram.html'))
print('  ✓ in_degree_histogram.html')

 ── Figure 4: Out-degree histogram ──────────────────────────────────────
fig4 = px.histogram(x=out_degrees, nbins=30,
    title='Circuit — Out-Degree Distribution', template='plotly_dark',
    labels={'x': 'Out-Degree', 'y': 'Count'})
fig4.write_html(str(FIGURE_DIR / 'out_degree_histogram.html'))
print('  ✓ out_degree_histogram.html')

 ── Figure 5: Candidate family comparison bar chart ─────────────────────
family_bar = pd.DataFrame([
    {'family': r['family'], 'best_circuit_size': r['best_size']}
    for r in family_results
])
fig5 = px.bar(family_bar, x='family', y='best_circuit_size',
    title='Best Circuit Size per Candidate Family', template='plotly_dark',
    labels={'best_circuit_size': 'Neurons', 'family': 'Triplet Family'},
    color='best_circuit_size', color_continuous_scale='Viridis')
fig5.write_html(str(FIGURE_DIR / 'family_comparison.html'))
print('  ✓ family_comparison.html')

 ── Figure 6: Circuit size by seed rank (winning family) ─────────────────
winning_result = next((r for r in family_results
    if r['family'] == f'{d1}-{d2}-{d3}'), None)
if winning_result and winning_result['seed_sizes']:
    seed_df = pd.DataFrame(winning_result['seed_sizes'])
    fig6 = px.line(seed_df, x='seed_rank', y='circuit_size',
        title=f'Circuit Size by Seed Rank — {d1}-{d2}-{d3}',
        template='plotly_dark',
        labels={'seed_rank': 'Seed Rank', 'circuit_size': 'Circuit Size'})
    fig6.write_html(str(FIGURE_DIR / 'circuit_size_by_seed.html'))
    print('  ✓ circuit_size_by_seed.html')

print('\n✓ All figures generated')


STEP 8 — GENERATING FIGURES
  ✓ circuit_network.html
  ✓ degree_histogram.html
  ✓ in_degree_histogram.html
  ✓ out_degree_histogram.html
  ✓ family_comparison.html
  ✓ circuit_size_by_seed.html

✓ All figures generated


In [17]:
 ── STEP 9 — SAVE FINAL CHECKPOINT ──────────────────────────────────────
print('=' * 60)
print('STEP 9 — SAVING CHECKPOINT')
print('=' * 60)

ckpt_out = CHECKPOINT_DIR / 'phase8_10_checkpoint.pkl'

checkpoint = {
    'best_mapping'       : best_mapping,
    'best_family'        : best_family,
    'best_seed_rank'     : best_seed_rank,
    'best_N'             : best_N,
    'best_edges'         : best_edges,
    'metrics'            : metrics,
    'submission_df'      : submission_df,
    'validation_results' : validation_results,
    'family_results'     : family_results,
     Record the search parameters so this run is fully reproducible.
    'search_parameters'  : {
        'TOP_SEEDS'             : TOP_SEEDS,
        'MAX_CANDIDATES_TO_TEST': MAX_CANDIDATES_TO_TEST,
        'TARGET_FAMILIES'       : TARGET_FAMILIES,
        'RANDOM_SEED'           : RANDOM_SEED,
    },
}

with open(ckpt_out, 'wb') as f:
    pickle.dump(checkpoint, f)

size_mb = ckpt_out.stat().st_size / 1e6
print(f'  Saved → {ckpt_out}  ({size_mb:.1f} MB)')
print('\n✓ phase8_10_checkpoint.pkl saved')


STEP 9 — SAVING CHECKPOINT
  Saved → /kaggle/working/results/checkpoints/phase8_10_checkpoint.pkl  (0.1 MB)

✓ phase8_10_checkpoint.pkl saved


In [18]:
 ── STEP 10 — FINAL REPORT ──────────────────────────────────────────────
print('=' * 60)
print('FINAL CONSERVED CIRCUIT')
print('=' * 60)
print()
print(f'  Winning Family          : {best_family[0]}-{best_family[1]}-{best_family[2]}')
print(f'  Seed Rank               : {best_seed_rank}')
print(f'  Neuron Count            : {best_N}')
print(f'  Edge Count              : {best_edges}')
print(f'  Density                 : {metrics["density"]}')
print()
print(f'  Exact Isomorphism G1/G2 : {validation_results["G1_vs_G2"]}')
print(f'  Exact Isomorphism G1/G3 : {validation_results["G1_vs_G3"]}')
print()
print(f'  Submission File         : {WORK_DIR}/submission.csv')
print(f'  Checkpoint              : {CHECKPOINT_DIR}/phase8_10_checkpoint.pkl')
print()
print('=' * 60)
print(' READY FOR FLYWIRE SUBMISSION')
print('=' * 60)


FINAL CONSERVED CIRCUIT

  Winning Family          : MAOL-FAFB-MCNS
  Seed Rank               : 9
  Neuron Count            : 161
  Edge Count              : 301
  Density                 : 0.011685

  Exact Isomorphism G1/G2 : True
  Exact Isomorphism G1/G3 : True

  Submission File         : /kaggle/working/submission.csv
  Checkpoint              : /kaggle/working/results/checkpoints/phase8_10_checkpoint.pkl

# READY FOR FLYWIRE SUBMISSION


In [19]:
import shutil
from pathlib import Path

WORK_DIR = Path("/kaggle/working")

 Create one zip containing everything generated by the notebook
zip_path = "/kaggle/working/flywire_submission_bundle.zip"

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir="/kaggle/working"
)

print(f"Created: {zip_path}")

Created: /kaggle/working/flywire_submission_bundle.zip
